# Australia 2025 F1 Telemetry Analysis

Comprehensive analysis of driver telemetry data from Australia 2025 qualifying and race.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Portable project paths (works for any clone location)
project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent
elif not (project_root / 'requirements.txt').exists() and (project_root.parent / 'requirements.txt').exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root))

from src.data_io import read_csv
from src.paths import (
    QUALIFYING_PROCESSED,
    QUALIFYING_RAW,
    RACE_RAW,
)

# Load qualifying data (prefer processed; fall back to raw or .csv.gz)
try:
    quali_df = read_csv(QUALIFYING_PROCESSED)
    print("✓ Loaded processed qualifying data (with acceleration & corners)")
except FileNotFoundError:
    quali_df = read_csv(QUALIFYING_RAW)
    print("✓ Loaded raw qualifying data")

# Load race data if available
try:
    race_df = read_csv(RACE_RAW)
    print("✓ Loaded race data")
except FileNotFoundError:
    race_df = None
    print("Note: Race data not available")

print(f"\nQualifying data loaded:")
print(f"  Shape: {quali_df.shape}")
print(f"  Columns: {quali_df.columns.tolist()}")
print(f"  Drivers: {quali_df['Driver'].nunique()}")

if race_df is not None:
    print(f"\nRace data loaded:")
    print(f"  Shape: {race_df.shape}")
    print(f"  Drivers: {race_df['Driver'].nunique()}")
else:
    print("\nRace data: Use raw race CSV for analysis")

## Qualifying Analysis

In [ ]:
# Best laps by driver in qualifying
if 'LapTime' not in quali_df.columns:
    print("LapTime column not found in qualifying data.")
else:
    lap_summary = (
        quali_df.dropna(subset=['LapTime'])
        .groupby(['Driver', 'LapNumber'], as_index=False)
        .agg(LapTime=('LapTime', 'first'))
    )
    lap_summary['LapTime_td'] = pd.to_timedelta(lap_summary['LapTime'], errors='coerce')
    lap_summary = lap_summary.dropna(subset=['LapTime_td'])

    quali_best_laps = (
        lap_summary.loc[lap_summary.groupby('Driver')['LapTime_td'].idxmin()]
        .assign(LapTime_seconds=lambda df: df['LapTime_td'].dt.total_seconds())
        .sort_values('LapTime_seconds')
    )

    print("Qualifying - Top 10 Best Laps:")
    print(quali_best_laps[['Driver', 'LapNumber', 'LapTime', 'LapTime_seconds']].head(10))

In [ ]:
# Throttle and brake application analysis - Qualifying
quali_telemetry_stats = quali_df.groupby('Driver').agg({
    'Throttle': ['mean', 'max', 'min'],
    'Brake': ['mean', 'max'],
    'Speed': ['mean', 'max'],
    'RPM': ['mean', 'max']
}).round(2)

print("\nQualifying - Telemetry Statistics (All drivers):")
print(quali_telemetry_stats)

## Race Analysis

## Corner & Acceleration Analysis (Qualifying)

In [ ]:
# Highest deceleration events from processed telemetry
if 'Corner' in quali_df.columns and 'Acceleration' in quali_df.columns:
    corner_events = (
        quali_df[quali_df['Corner'].notna()]
        .groupby(['Driver', 'Corner'], as_index=False)
        .agg(
            Min_Speed=('Speed', 'min'),
            Max_Deceleration=('Acceleration', 'min'),
            Max_Acceleration_Exit=('Acceleration', 'max'),
        )
    )

    high_decel = corner_events.dropna(subset=['Max_Deceleration']).sort_values('Max_Deceleration')

    print("Corner Events - Highest Deceleration")
    print("=" * 70)
    print(f"\nTotal corner events: {len(corner_events)}")
    print(f"Unique corners: {corner_events['Corner'].nunique()}")
    print(f"Drivers: {corner_events['Driver'].nunique()}")
    print("\nHighest Deceleration Events:")
    print(high_decel.head(15).to_string(index=False))
else:
    print("Corner and acceleration columns not found. Load processed telemetry first.")

In [ ]:
# Corner data for a specific driver - Max Verstappen
driver = 'VER'

if 'Corner' in quali_df.columns and 'Acceleration' in quali_df.columns:
    driver_corners = quali_df[(quali_df['Driver'] == driver) & (quali_df['Corner'].notna())]
    
    if len(driver_corners) > 0:
        print(f"\n{driver} - Acceleration/Deceleration at Corners")
        print("="*70)
        
        # Get corners
        for corner in sorted(driver_corners['Corner'].unique()):
            corner_data = driver_corners[driver_corners['Corner'] == corner]
            
            print(f"\n{corner}:")
            print(f"  Data points: {len(corner_data)}")
            print(f"  Min speed: {corner_data['Speed'].min():.1f} km/h")
            print(f"  Max speed: {corner_data['Speed'].max():.1f} km/h")
            print(f"  Max deceleration: {corner_data['Acceleration'].min():.2f} m/s²")
            print(f"  Max acceleration: {corner_data['Acceleration'].max():.2f} m/s²")
            print(f"  Avg throttle: {corner_data['Throttle'].mean():.1f}%")
            print(f"  Avg brake: {corner_data['Brake'].mean():.1f}%")
            
            # Show sample telemetry
            sample = corner_data[['TimeInLap', 'Speed', 'Throttle', 'Brake', 'Acceleration', 'Accel_Type']].head(5)
            print(f"\n  Sample telemetry:")
            print(sample.to_string(index=False))
    else:
        print("Corner data not yet processed. Run process_telemetry.py first.")
else:
    print("Acceleration and Corner columns not found. Load processed data.")

In [ ]:
# Visualization: Acceleration profiles by driver at Turn 1
if 'Corner' in quali_df.columns and 'Acceleration' in quali_df.columns:
    turn1_data = quali_df[quali_df['Corner'] == 'Turn 1'].copy()
    
    if len(turn1_data) > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Deceleration into corner
        ax1 = axes[0]
        decel_data = turn1_data[turn1_data['Accel_Type'] == 'Deceleration'].groupby('Driver')['Acceleration'].min()
        decel_data.sort_values().tail(10).plot(kind='barh', ax=ax1, color='red', alpha=0.7)
        ax1.set_xlabel('Max Deceleration (m/s²)')
        ax1.set_title('Turn 1 - Highest Deceleration by Driver')
        ax1.grid(axis='x', alpha=0.3)
        
        # Acceleration exiting corner
        ax2 = axes[1]
        accel_data = turn1_data[turn1_data['Accel_Type'] == 'Acceleration'].groupby('Driver')['Acceleration'].max()
        accel_data.sort_values(ascending=False).head(10).plot(kind='barh', ax=ax2, color='green', alpha=0.7)
        ax2.set_xlabel('Max Acceleration (m/s²)')
        ax2.set_title('Turn 1 - Highest Acceleration Exit by Driver')
        ax2.grid(axis='x', alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        print("Chart shows acceleration/deceleration profiles at Turn 1 for qualifying")

In [ ]:
# Race telemetry statistics
if race_df is None:
    print("Race data not available. Run extract_australia_telemetry.py to fetch it.")
else:
    race_telemetry_stats = race_df.groupby('Driver').agg({
        'Throttle': ['mean', 'max'],
        'Brake': ['mean', 'max'],
        'Speed': ['mean', 'max'],
        'RPM': ['mean', 'max'],
        'LapNumber': 'max',
    }).round(2)

    print("Race - Telemetry Statistics:")
    print(race_telemetry_stats)

In [ ]:
# Single driver deep dive - Max Verstappen
if race_df is None:
    print("Race data not available for driver deep dive.")
else:
    ver_race = race_df[race_df['Driver'] == 'VER']

    print("Max Verstappen - Race Performance:")
    print(f"  Total telemetry points: {len(ver_race):,}")
    print(f"  Laps completed: {ver_race['LapNumber'].max()}")
    print(f"  Avg throttle: {ver_race['Throttle'].mean():.2f}%")
    print(f"  Max throttle: {ver_race['Throttle'].max():.2f}%")
    print(f"  Avg brake: {ver_race['Brake'].mean():.2f}%")
    print(f"  Max brake: {ver_race['Brake'].max():.2f}%")
    print(f"  Avg speed: {ver_race['Speed'].mean():.2f} km/h")
    print(f"  Max speed: {ver_race['Speed'].max():.2f} km/h")

## Data Exploration

In [ ]:
# Check data structure
print("Qualifying data columns:")
print(quali_df.columns.tolist())
print(f"\nData types:")
print(quali_df.dtypes)

In [ ]:
# Sample data view
print("Sample qualifying data:")
print(quali_df[quali_df['Driver'] == 'VER'].head(10)[['Driver', 'LapNumber', 'TimeInLap', 'Speed', 'Throttle', 'Brake', 'X', 'Y', 'Z']])